In [1]:
import os
import csv
import rclpy
from rclpy.node import Node
from std_msgs.msg import Int8MultiArray, UInt8

output_csv = str(os.environ.get('HOME')) + "/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/TE02_001/telejoy_latency.csv"

def create_csv():
    if not os.path.exists(os.path.dirname(output_csv)):
        os.makedirs(os.path.dirname(output_csv))

    with open(output_csv, 'w', newline='') as csvfile:
        fieldnames = ['timestamp', 'latency_ms', 'new_status_value']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

def write_csv(timestamp, latency_ms, new_status):
    with open(output_csv, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        # Format: [timestamp, latency_ms, new_status]
        row_data = [f"{timestamp.sec}.{timestamp.nanosec:09d}", f"{latency_ms:.2f}", new_status]
        writer.writerow(row_data)

create_csv()


In [4]:
class JoyLatencyTester(Node):
    def __init__(self):
        super().__init__('joy_latency_tester')

        self.pub = self.create_publisher(Int8MultiArray, '/telejoy', 10)
        self.sub = self.create_subscription(
            UInt8, '/statusjoy', self.status_cb, 10)

        self.initial_status = None
        self.trigger_time = None
        self.state = 'WAITING_FOR_BASELINE'

        # Trigger the change
        self.target_command = [0, 0, 0, 0, 4, 0]
        # Establish a baseline
        self.current_command = [0, 0, 0, 0, 0, 0]

        self.timer = self.create_timer(0.1, self.publish_command)

    def publish_command(self):
        msg = Int8MultiArray()
        msg.data = self.current_command
        self.pub.publish(msg)

    def status_cb(self, msg):
        current_status = msg.data

        if self.state == 'WAITING_FOR_BASELINE':
            self.initial_status = current_status
            self.get_logger().info(
                f"Baseline status established: {self.initial_status}")

            self.get_logger().info(
                f"Sending target command: {self.target_command}")
            self.current_command = self.target_command

            self.trigger_time = self.get_clock().now()
            self.state = 'MEASURING'

        elif self.state == 'MEASURING':
            if current_status != self.initial_status:
                arrival_time = self.get_clock().now()

                # Calculate latency
                latency_ns = (arrival_time - self.trigger_time).nanoseconds
                latency_ms = (latency_ns) / 1_000_000.0

                self.get_logger().info(
                    f"Status changed to {current_status}! Latency: {latency_ms/2:.2f} ms")

                # Log to CSV using the exact time the trigger was sent
                write_csv(self.trigger_time.to_msg(),
                          latency_ms/2, current_status)

                self.state = 'DONE'

                # Signal the node to shut down to prevent endless logging
                raise SystemExit

In [7]:
# Initialize ROS 2 if not already running
if not rclpy.ok():
    rclpy.init()

tester_node = JoyLatencyTester()

try:
    # Spin blocks until SystemExit is raised in the status_cb
    rclpy.spin(tester_node)
except SystemExit:
    tester_node.get_logger().info("Test complete. Data written to CSV.")
except KeyboardInterrupt:
    tester_node.get_logger().info("Test manually aborted.")
finally:
    # Clean up so you can run the cell again immediately
    tester_node.destroy_node()

[INFO] [1783348109.964846917] [joy_latency_tester]: Baseline status established: 1
[INFO] [1783348109.967353577] [joy_latency_tester]: Sending target command: [0, 0, 0, 0, 4, 0]
[INFO] [1783348109.971013429] [joy_latency_tester]: Status changed to 10! Latency: 0.22 ms
[INFO] [1783348111.637052350] [joy_latency_tester]: Test complete. Data written to CSV.
